# AdaBoost: From First Principles

AdaBoost, short for Adaptive Boosting, is a sequential ensemble method. It combines weak classifiers into a stronger classifier by changing the training-example weights after each round: examples the current learner gets wrong become more influential in the next round.

This lesson focuses on the classic binary AdaBoost algorithm with labels in $\{-1,+1\}$. Multi-class versions and implementation details differ, so the exact formulas below should not be assumed for every estimator named AdaBoost.

## Why AdaBoost Is Useful

A single simple classifier may not separate the classes well. Boosting builds an additive model in stages: each new classifier is trained with a changed emphasis so that difficult examples can influence later decisions more strongly.

This is different from bagging. Bagging trains models largely independently on resampled data and combines them to reduce variance. AdaBoost trains sequentially, with each round depending on the previous round's errors.

The algorithm has an optimization interpretation: in its classic form, the reweighting process corresponds to minimizing an exponential loss. The example weights are a mechanism for focusing the next weak learner; they are not probabilities that examples are mislabeled.

## Weak Learners

A weak learner is a model that can perform only modestly well on the current weighted training distribution, yet can still do better than random guessing. AdaBoost's classic theoretical setup assumes that the learner can find such a model at each round.

A common choice is a decision stump: a tree with one split and a prediction at each leaf. A stump may be too simple to model a complicated boundary by itself, but different stumps can capture different useful rules. Their weighted combination can express a richer classifier.

The learner must use the example weights when fitting or selecting splits. Ignoring those weights would mean later rounds did not actually pay extra attention to examples that previous learners missed.

## The AdaBoost Cycle

For binary classification, encode labels and weak-learner predictions as $y_i,h_t(\mathbf{x}_i)\in\{-1,+1\}$. AdaBoost keeps a distribution $D_t(i)$ over training examples, initially $D_1(i)=1/n$. At each round it fits a weak learner using those weights, then calculates its weighted error:

$$
\epsilon_t = \sum_{i=1}^{n}D_t(i)\,\mathbb{1}\left[h_t(\mathbf{x}_i)\ne y_i\right]
$$

The learner's vote weight is based on whether it performs better than random guessing. AdaBoost then increases the relative weights of misclassified examples and decreases the relative weights of correctly classified ones. The next learner is fit against this updated distribution.

The learners are added sequentially, not independently: changing one round changes the weights seen by later rounds.

## Learner Weight and Example-Weight Update

For the classic binary algorithm, a learner with weighted error $\epsilon_t<1/2$ receives vote weight:

$$
\alpha_t = \frac{1}{2}\ln\left(\frac{1-\epsilon_t}{\epsilon_t}\right)
$$

A lower weighted error gives a larger positive vote. If the learner is exactly at chance, $\alpha_t=0$ and it adds no vote. If its error is worse than chance, the weight is negative; the standard weak-learning assumption has failed, so an implementation may reject the round or invert the learner when that is valid. If the learner has zero error, this expression diverges, and implementations typically stop or handle the perfect fit specially.

The example weights are updated and normalized:

$$
D_{t+1}(i) = \frac{D_t(i)\exp\left(-\alpha_t y_i h_t(\mathbf{x}_i)\right)}{Z_t}
$$

$Z_t$ is the normalizing constant that makes the new weights sum to 1. If the example is correct, $y_i h_t(\mathbf{x}_i)=+1$ and its unnormalized weight is multiplied by $e^{-\alpha_t}$. If incorrect, the product is $-1$ and its weight is multiplied by $e^{+\alpha_t}$. Thus, mistakes receive greater relative weight for the next round.

## Worked Example: Reweighting Five Examples

Suppose five examples start with equal weights of $0.2$, and a stump misclassifies one example. Its weighted error is $\epsilon=0.2$, so its vote weight is:

$$
\alpha = \frac{1}{2}\ln\left(\frac{0.8}{0.2}\right)=\frac{1}{2}\ln(4)=\ln(2)\approx0.693
$$

Before normalization, the mistaken example is multiplied by $e^{\alpha}=2$, so its weight becomes $0.4$. Each of the four correct examples is multiplied by $e^{-\alpha}=1/2$, so each becomes $0.1$. The weights sum to $0.8$, so normalization gives the mistaken example weight $0.5$ and each correct example weight $0.125$.

The next learner therefore sees the one previously missed example as half of the total weighted distribution, rather than one fifth. AdaBoost still considers every example; it changes their relative influence.

## Final Prediction

After $M$ rounds, the ensemble combines the weak predictions using their vote weights:

$$
F(\mathbf{x}) = \sum_{t=1}^{M}\alpha_t h_t(\mathbf{x}),
\qquad
\hat{y}(\mathbf{x}) = \operatorname{sign}(F(\mathbf{x}))
$$

A learner with lower weighted error contributes more strongly. The sum $F(\mathbf{x})$ is a real-valued margin score; its sign gives the binary class. It is not automatically a calibrated class probability. A zero score is a tie and requires an implementation-specific convention.

## Strengths, Limitations, and Evaluation

AdaBoost combines simple learners into a flexible classifier and gives an interpretable account of how each round's vote is weighted. With decision-tree stumps, it can model some nonlinear decision boundaries without requiring feature scaling.

Its reweighting is also a weakness. A mislabeled point, outlier, or inherently ambiguous example may repeatedly receive high weight, causing later learners to focus on noise. The sequential dependency makes rounds harder to parallelize than independently trained bagged models. Performance can also be affected by class imbalance and by a base learner that is too restricted or too complex.

Tune the number of learners, learning rate, and base-estimator complexity using validation data. More rounds can improve the fit but can also amplify noise. Evaluate with metrics appropriate to the class balance and error costs, such as precision, recall, F1, ROC-AUC, or PR-AUC. The ensemble's vote score is not automatically a calibrated probability; calibrate separately if probability estimates are needed.

AdaBoost is useful when a simple base learner is appropriate and sequentially correcting errors helps. Compare it with other methods rather than assuming boosting will always improve generalization.

## Variants and Practical Choices

The equations in this lesson describe the classic discrete binary AdaBoost setup. Multi-class classification extends the approach with variants such as SAMME, while regression uses different variants such as AdaBoost.R2; their losses and update rules are not identical to the binary formulas above. Check the estimator documentation for the specific algorithm implemented by your library and version.

Common choices include the number of boosting rounds, the learning rate, and the base estimator. A smaller learning rate shrinks each learner's contribution and often calls for more rounds, but it does not guarantee a better model. The base learner's complexity controls the patterns each round can fit. Tune these choices on training/validation data, and reserve the test set for final evaluation.

The classic binary formulas assume a weak learner with weighted error below $1/2$. In practice, inspect validation performance and stop adding complexity when it no longer helps; do not interpret a larger ensemble as automatically stronger.